# Pizza Hut Chatbot with Gradio
A conversational chatbot for Pizza Hut that can take orders, answer menu questions, and handle customer queries — built with LangChain and Gradio.

In [1]:
pip install langchain langchain-ollama gradio --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Define the Pizza Hut Menu & System Prompt

In [2]:
SYSTEM_PROMPT = """
You are a friendly and helpful chatbot for Pizza Hut.
Your job is to help customers with:
- Browsing the menu
- Taking orders
- Answering questions about ingredients and allergens
- Providing information about deals and offers
- Estimating delivery time (30-45 minutes)

MENU:
--- PIZZAS ---
- Margherita (S $8 / M $12 / L $16) — tomato sauce, mozzarella, basil
- Pepperoni (S $10 / M $14 / L $18) — tomato sauce, mozzarella, pepperoni
- BBQ Chicken (S $11 / M $15 / L $19) — BBQ sauce, chicken, onions, mozzarella
- Veggie Supreme (S $10 / M $14 / L $18) — tomato sauce, bell peppers, mushrooms, olives
- Meat Lovers (S $12 / M $16 / L $20) — pepperoni, sausage, bacon, beef

--- SIDES ---
- Garlic Bread $4
- Chicken Wings (6 pcs) $8
- Caesar Salad $6

--- DRINKS ---
- Coca-Cola $2
- Sprite $2
- Water $1

--- DEALS ---
- Family Deal: 2 Large Pizzas + Garlic Bread + 2 Drinks = $35
- Lunch Special (Mon-Fri 11am-3pm): Any Medium Pizza + Drink = $12

Always be polite, confirm the order at the end, and ask for delivery address if the customer wants delivery.
If asked something outside of Pizza Hut topics, politely redirect the conversation.
"""

## 2. Set Up the LangChain Chat Model

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOllama(model='llama3.2', temperature=0.7)

def get_response(user_message, history):
    """
    Takes the user message and full chat history (list of dicts),
    returns the assistant's reply.
    """
    messages = [SystemMessage(content=SYSTEM_PROMPT)]

    # Add conversation history — Gradio 6 uses {'role': ..., 'content': ...} format
    for msg in history:
        if msg['role'] == 'user':
            messages.append(HumanMessage(content=msg['content']))
        else:
            messages.append(AIMessage(content=msg['content']))

    # Add current user message
    messages.append(HumanMessage(content=user_message))

    response = llm.invoke(messages)
    return response.content

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 3. Test the Chatbot Without UI

In [4]:
history = []

questions = [
    'Hi! What pizzas do you have?',
    'I want a large Pepperoni and Garlic Bread.',
    'Do you have any deals?'
]

for q in questions:
    reply = get_response(q, history)
    history.append({'role': 'user', 'content': q})
    history.append({'role': 'assistant', 'content': reply})
    print(f'Customer : {q}')
    print(f'Bot      : {reply}\n')

Customer : Hi! What pizzas do you have?
Bot      : We have a variety of delicious pizzas to choose from. Here are our current pizza options:

* Margherita: a classic combination of tomato sauce, mozzarella, and fresh basil
* Pepperoni: a popular choice topped with pepperoni, tomato sauce, and mozzarella
* BBQ Chicken: a sweet and tangy combination of BBQ sauce, chicken, onions, and mozzarella
* Veggie Supreme: a vegetarian's dream with bell peppers, mushrooms, olives, and tomato sauce
* Meat Lovers: a hearty combination of pepperoni, sausage, bacon, and beef

We also have a range of crust sizes: Small (S), Medium (M), and Large (L). Would you like me to tell you more about any of these options or help you narrow down your choices?

Customer : I want a large Pepperoni and Garlic Bread.
Bot      : One Large Pepperoni Pizza with garlic bread coming right up.

Would you like to add any sides or drinks to your order? We have a range of options, including our Caesar Salad, Chicken Wings, Coc

## 4. Launch the Gradio Chatbot UI

In [6]:
import gradio as gr

def chat(user_message, history):
    reply = get_response(user_message, history)
    history.append({'role': 'user', 'content': user_message})
    history.append({'role': 'assistant', 'content': reply})
    return '', history

with gr.Blocks(title='Pizza Hut Chatbot') as demo:

    gr.Markdown("""
    # 🍕 Pizza Hut Chatbot
    Welcome! I can help you browse our menu, place an order, or answer any questions.
    """)

    chatbot = gr.Chatbot(height=450, label='Pizza Hut Assistant')
    msg = gr.Textbox(placeholder='Type your message here...', label='You')
    clear = gr.Button('Clear Chat')

    msg.submit(chat, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], outputs=chatbot)

demo.launch(theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
